# 04 — Pretrain một mô hình từ đầu

    Chạy độc lập trên Kaggle T4 sau review giai đoạn 2. `RUN_MODE='inspect'` là mặc định, không train. `smoke` chạy tối đa 2 update bằng model nhỏ trên shard thật; `production` mới khởi tạo khoảng 36M tham số và chạy tối đa 1 epoch pretrain. Luôn dùng tokenizer và shard cùng một lần chuẩn bị dữ liệu. Local: `uv sync --group model --group train --group data --group notebook`. Kaggle: xác minh torch CUDA cài sẵn; cài thiếu `tensorboard numpy` rồi khởi động lại kernel. Không cài lại torch bằng wheel CPU.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import torch

REPO_URL = 'https://github.com/ngochhuy/llm-reasoning.git'
REPO_REF = ''  # Kaggle: đặt commit SHA đã review; không dùng main di động.
ON_KAGGLE = Path('/kaggle/working').exists()
REPO_DIR = Path('/kaggle/working/llm-reasoning-repo') if ON_KAGGLE else next(
    (p for p in (Path.cwd(), Path.cwd().parent) if (p/'src'/'llm_reasoning').exists()), Path.cwd())
if ON_KAGGLE:
    if len(REPO_REF) < 40:
        raise ValueError('Đặt REPO_REF bằng commit SHA 40 ký tự đã review trước khi chạy Kaggle')
    if not (REPO_DIR/'src'/'llm_reasoning').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
print('commit:', CODE_COMMIT, 'dirty:', DIRTY, 'torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
sys.path.insert(0, str(REPO_DIR/'src'))
os.environ['PYTHONPATH'] = str(REPO_DIR/'src') + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
from dataclasses import asdict, replace
from llm_reasoning.checkpoint import file_sha256, load_checkpoint, save_checkpoint
from llm_reasoning.config import make_config
from llm_reasoning.data_stream import ResumableBatchStream
from llm_reasoning.model import TransformerLM
from llm_reasoning.tokenizer import ByteBPETokenizer
from llm_reasoning.training import Trainer, TrainingLogger
from llm_reasoning.training_data import PretrainBatches
import time

RUN_MODE = 'inspect'  # inspect | smoke | production; bật production sau review.
DATA_ROOT = Path('/kaggle/input/llm-reasoning-data/pretrain') if ON_KAGGLE else REPO_DIR/'artifacts/data/pretrain'
OUTPUT_ROOT = Path('/kaggle/working/pretrain_run') if ON_KAGGLE else REPO_DIR/'artifacts/nb04_pretrain'
RESUME_FROM = None  # Path đến latest.pt đã lưu bền vững; giữ đúng dataset, commit và thiết bị.
MAX_UPDATES_THIS_SESSION = None  # Pilot: 200-500, rồi resume cùng checkpoint và total_steps.
if RUN_MODE not in ('inspect', 'smoke', 'production'):
    raise ValueError('RUN_MODE không hợp lệ')
if RUN_MODE == 'production' and (not ON_KAGGLE or not torch.cuda.is_available() or DIRTY):
    raise RuntimeError('Production yêu cầu Kaggle CUDA và commit sạch đã chốt')
if RESUME_FROM is not None and RUN_MODE != 'production':
    raise ValueError('Resume chỉ dùng cho cùng production run')
tokenizer_path = DATA_ROOT/'tokenizer.json'
manifest_path = DATA_ROOT/'manifest.json'
tokenizer = ByteBPETokenizer.load(tokenizer_path)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest['tokenizer_sha256'] != file_sha256(tokenizer_path) or manifest['vocab_size'] != tokenizer.vocab_size:
    raise ValueError('Tokenizer hash/vocab không khớp manifest')
if manifest['block_size'] != 512 and RUN_MODE == 'production':
    raise ValueError('Production yêu cầu pretrain block size 512')
if RUN_MODE == 'production':
    for source, target in (('fineweb', 80_000_000), ('math', 20_000_000)):
        if manifest['sources'][source]['shards']['train']['written_tokens'] < target - 512:
            raise ValueError(f'{source} chưa đạt ngân sách token pretrain; tạo lại dữ liệu trước GPU run')
print('Mode:', RUN_MODE, 'tokenizer SHA:', file_sha256(tokenizer_path), 'vocab:', tokenizer.vocab_size)
print('Source train tokens:', {s: d['shards']['train']['written_tokens'] for s,d in manifest['sources'].items()})

## Kiểm tra batch và tính lịch train

    Tất cả batch đọc từ shard uint16 do notebook 02 tạo. Fingerprint bao gồm manifest, shard và cách chia batch; resume từ dữ liệu khác sẽ bị từ chối. `smoke` chỉ dùng vài batch và checkpoint thử, không được dùng làm pretrain chính.

In [ ]:
if RUN_MODE != 'inspect':
    config = make_config('smoke' if RUN_MODE == 'smoke' else 'target')
    train_cfg = config.train
    if RUN_MODE == 'smoke':
        train_cfg = replace(train_cfg, pretrain_seq_len=manifest['block_size'],
                            micro_batch_size=1, gradient_accumulation_steps=1)
        model_cfg = replace(config.model, max_context_tokens=max(1024, manifest['block_size']))
    else:
        model_cfg = config.model
    if train_cfg.pretrain_seq_len != manifest['block_size']:
        raise ValueError('Block size khác pretrain_seq_len')
    train_data = PretrainBatches(DATA_ROOT, 'train', batch_size=train_cfg.micro_batch_size,
                                 max_batches=4 if RUN_MODE == 'smoke' else None)
    stream = ResumableBatchStream(train_data, fingerprint=train_data.fingerprint,
                                  seed=config.data.seed, shuffle=True)
    total_steps = (len(train_data) + train_cfg.gradient_accumulation_steps - 1) // train_cfg.gradient_accumulation_steps
    device = 'cuda' if RUN_MODE == 'production' else 'cpu'
    if RUN_MODE == 'smoke':
        train_cfg = replace(train_cfg, precision='fp32')
    print('Microbatches:', len(train_data), 'total steps:', total_steps,
          'batch:', train_data[0]['input_ids'].shape, 'fingerprint:', train_data.fingerprint)
else:
    print('Inspect hoàn tất; chưa khởi tạo model/trainer, chưa train.')

## Train, checkpoint và validation

    Chạy lại từ phiên mới: tải `latest.pt`, cùng tokenizer/shard vào Kaggle Input; đặt `RESUME_FROM` và giữ nguyên cấu hình batch/dataset. `MAX_UPDATES_THIS_SESSION` dừng ở biên update; có thể dùng 200–500 update để đo pilot, rồi resume từ `latest.pt`. Checkpoint cuối luôn lưu sau mỗi phiên. Copy toàn bộ output sang Kaggle Dataset/Notebook Version trước khi phiên kết thúc.

In [ ]:
if RUN_MODE != 'inspect':
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    checkpoint_path = OUTPUT_ROOT/'latest.pt'
    log_dir = OUTPUT_ROOT/'logs'
    if RESUME_FROM is None:
        if checkpoint_path.exists() or (log_dir/'metrics.csv').exists():
            raise FileExistsError('Output cũ tồn tại; chọn thư mục mới hoặc đặt RESUME_FROM')
        torch.manual_seed(config.data.seed)
        model = TransformerLM(model_cfg, tokenizer.vocab_size).to(device)
        trainer = Trainer(model, train_cfg, stage='pretrain', total_steps=total_steps)
    else:
        trainer = load_checkpoint(RESUME_FROM, stream, tokenizer_path=tokenizer_path,
                                  device=device, expected_stage='pretrain')
        if trainer.total_steps != total_steps or trainer.config != train_cfg:
            raise ValueError('Config hoặc total_steps thay đổi sau resume')
    kind = 'smoke' if RUN_MODE == 'smoke' else 'training'
    run_manifest = dict(stage='pretrain', kind=kind, code_commit=CODE_COMMIT,
                        tokenizer_sha256=file_sha256(tokenizer_path),
                        data_fingerprint=train_data.fingerprint, model_config=asdict(model_cfg),
                        train_config=asdict(train_cfg), total_steps=total_steps, device=device)
    run_manifest_path = OUTPUT_ROOT/'run_manifest.json'
    if run_manifest_path.exists():
        if json.loads(run_manifest_path.read_text(encoding='utf-8')) != run_manifest:
            raise ValueError('Run manifest thay đổi; không thể resume cùng output')
    else:
        run_manifest_path.write_text(json.dumps(run_manifest, indent=2)+'\n', encoding='utf-8')
    with TrainingLogger(log_dir, resume_step=trainer.state.attempted_steps if RESUME_FROM else None) as logger:
        trainer.logger = logger
        last_saved = time.monotonic()
        def periodic(current):
            global last_saved
            if time.monotonic() - last_saved >= train_cfg.checkpoint_interval_minutes * 60:
                save_checkpoint(checkpoint_path, current, stream,
                                tokenizer_path=tokenizer_path, checkpoint_kind=kind)
                last_saved = time.monotonic()
        limit = 2 if RUN_MODE == 'smoke' else MAX_UPDATES_THIS_SESSION
        rows = trainer.train_epoch(stream, max_updates=limit, on_update=periodic)
        save_checkpoint(checkpoint_path, trainer, stream,
                        tokenizer_path=tokenizer_path, checkpoint_kind=kind)
        print('New updates:', len(rows), 'total:', trainer.state.attempted_steps,
              'processed tokens:', trainer.state.processed_tokens, 'checkpoint:', checkpoint_path)
        if stream.cursor == len(stream.batches):
            validation = PretrainBatches(DATA_ROOT, 'validation', batch_size=train_cfg.micro_batch_size,
                                         max_batches=2 if RUN_MODE == 'smoke' else None)
            print('Validation:', trainer.evaluate(validation))
    print('Train xong phiên này. Lưu checkpoint và logs trước khi đóng phiên Kaggle.')